# Multi-Agent Orchestration for RAG Systems
# Step 2. Specialized Agents for Orchestrated RAG

*Assignee: Alla* - *Review:*

This notebook implements the specialized-agent layer for a modular RAG system. It defines role-based agents with a shared `AgentState` contract and keeps retriever artifacts reusable for later orchestration experiments.

The design is intentionally decoupled from routing, so external orchestrators can apply strategies such as Parallel+Fusion, Sequential Waterfall, or Confidence-Based Routing without changing agent internals.



In [67]:
# Colab setup
%pip install -q langchain-core langchain-community langchain-huggingface chromadb \
  rank-bm25 langdetect nltk sentence-transformers


In [68]:
import os
import re
import json
import pickle
import random
import pathlib
from dataclasses import dataclass
from collections import defaultdict
from typing import Any

import numpy as np
import nltk
from langdetect import detect

from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

STOP_EN = set(nltk.corpus.stopwords.words('english'))
STOP_DE = set(nltk.corpus.stopwords.words('german'))

print('Setup complete.')



Setup complete.


In [69]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1.1 Setup

Requirements:
- Add `HF_TOKEN` to Colab Secrets (https://huggingface.co/settings/tokens).
- Ensure Google Drive contains:
  - `/content/drive/MyDrive/Adv_GenAI/benchmark`
  - `/content/drive/MyDrive/Adv_GenAI/storage`

Scope selection:
- `EVAL_SCOPE = 'full_corpus'` or `EVAL_SCOPE = 'subsample'`
- For orchestration comparison, `full_corpus` is recommended.



In [70]:
# Paths (edit PROJECT_ROOT if needed)
# PROJECT_ROOT must be the folder that contains benchmark/ and storage/
CANDIDATE_ROOTS = [
    pathlib.Path('/content/drive/MyDrive/Adv_GenAI'),
    pathlib.Path('/content/drive/MyDrive/advanced-genai-26/baseline/advanced_genAI-main/data'),
    pathlib.Path('/content/drive/MyDrive/advanced_genAI-main/data'),
]

def looks_like_project_root(p: pathlib.Path) -> bool:
    return (p / 'benchmark').exists() and (p / 'storage').exists()

PROJECT_ROOT = None
for c in CANDIDATE_ROOTS:
    if looks_like_project_root(c):
        PROJECT_ROOT = c
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not auto-detect project root. Set PROJECT_ROOT manually.')

PROJECT_ROOT = PROJECT_ROOT.resolve()
print('PROJECT_ROOT =', PROJECT_ROOT)

# Choose scope: 'subsample' or 'full_corpus'
EVAL_SCOPE = 'full_corpus'
assert EVAL_SCOPE in {'subsample', 'full_corpus'}
print('EVAL_SCOPE =', EVAL_SCOPE)

if EVAL_SCOPE == 'subsample':
    PATH_BM25_PICKLE = PROJECT_ROOT / 'storage/subsample/retrieval_downstream/bm25_fixed_qe.pkl'
    if not PATH_BM25_PICKLE.exists():
        PATH_BM25_PICKLE = PROJECT_ROOT / 'storage/subsample/retrieval/fixed_size_chunk/bm25_retriever.pkl'
    PATH_DENSE_INDEX = PROJECT_ROOT / 'storage/subsample/vectordb_dense/fixed_e5'
    PATH_GRAG_ROOT = PROJECT_ROOT / 'storage/subsample/retrieval_graph'
    PATH_CHUNK_PKL = PROJECT_ROOT / 'storage/subsample/Lang_norm/fixed_size_chunk/docs_fixed_norm.pkl'
else:
    PATH_BM25_PICKLE = PROJECT_ROOT / 'storage/full_corpus/retrieval/fixed_size_chunk/bm25_retriever_full.pkl'
    PATH_DENSE_INDEX = PROJECT_ROOT / 'storage/full_corpus/vectordb_dense/fixed_e5'
    PATH_GRAG_ROOT = PROJECT_ROOT / 'storage/full_corpus/retrieval_graph'
    PATH_CHUNK_PKL = PROJECT_ROOT / 'storage/full_corpus/Lang_norm/fixed_size_chunk/docs_fixed_norm.pkl'

required = [PATH_BM25_PICKLE, PATH_DENSE_INDEX, PATH_GRAG_ROOT, PATH_CHUNK_PKL]
for p in required:
    if not p.exists():
        raise FileNotFoundError(f'Missing required path: {p}')

print('All required retrieval artifacts found.')



PROJECT_ROOT = /content/drive/MyDrive/Adv_GenAI
EVAL_SCOPE = full_corpus
All required retrieval artifacts found.


## 2. Multi-Agent Architecture (Specialized Roles)

This is the main implementation section.

Implemented roles:
- Query Understanding Agent
- Retriever Agents (`BM25`, `Dense`, `GraphRAG`)
- Fusion Agent (merge + deduplicate)
- Re-Ranker Agent
- Answer Synthesizer Agent
- Critic Agent (grounding check + re-retrieval trigger)

All agents read/write a shared `AgentState`, making routing strategies interchangeable.



## 2.1 Retrieval Component: BM25 Adapter

Loads BM25 artifacts with a compatibility wrapper and exposes a stable `search(query, top_k)` interface across artifact variants.

**Retriever Compatibility Note**
Some BM25 artifacts were serialized with custom classes (e.g., `BilingualBM25` or `QEBM25`). During `pickle.load(...)`, Python must resolve those class definitions to reconstruct the object, even if we later access the retriever only through `BM25RetrieverAdapter`.

For this reason, these class definitions are intentionally kept in the notebook as deserialization shims for cross-version artifact compatibility.




In [71]:
# Robust BM25 loader for both subsample and full-corpus pickle formats
class BilingualBM25:
    """Compatibility class for notebook pickles."""

    def _rank_lang(self, q: str, lang: str, k: int):
        # Subsample-style object: self.bm25 + self.docs_by_lang
        try:
            q_tokens = nltk.word_tokenize(q)
        except Exception:
            q_tokens = q.split()
        scores = self.bm25[lang].get_scores(q_tokens)
        idx = np.argsort(scores)[::-1][:k]
        hits = []
        for i in idx:
            d = self.docs_by_lang[lang][i]
            d.metadata['bm25_score'] = float(scores[i])
            hits.append(d)
        return hits

    def _get_docs_with_scores(self, ret, qq, top_k):
        # Full-corpus-style object: self.retrievers
        if hasattr(ret, 'get_relevant_documents_with_scores'):
            try:
                return ret.get_relevant_documents_with_scores(qq, k=top_k)
            except Exception:
                pass

        if hasattr(ret, 'vectorizer') and hasattr(ret, 'docs'):
            try:
                toks = qq.lower().split()
                scores = ret.vectorizer.get_scores(toks)
                ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:top_k]
                return [(ret.docs[idx], float(score)) for idx, score in ranked]
            except Exception:
                pass

        if hasattr(ret, 'invoke'):
            try:
                old_k = getattr(ret, 'k', None)
                if old_k is not None:
                    ret.k = top_k
                docs = ret.invoke(qq)
                if old_k is not None:
                    ret.k = old_k
                return [(d, d.metadata.get('score', 0.0)) for d in docs[:top_k]]
            except Exception:
                pass

        return []

    def search(self, query: str, top_k: int = 100):
        # Route to correct behavior based on available attributes
        if hasattr(self, 'bm25') and hasattr(self, 'docs_by_lang'):
            src = detect(query) if query.strip() else 'en'
            src = src if src in ('en', 'de') else 'en'
            bag = []
            translator = getattr(self, 'translator', None)
            for lang in ('en', 'de'):
                q_lang = translator.translate(query, lang) if translator and lang != src else query
                bag.extend(self._rank_lang(q_lang, lang, top_k))

            best = {}
            for d in bag:
                uid = d.metadata.get('chunk_id') or d.metadata.get('record_id')
                if uid not in best or d.metadata['bm25_score'] > best[uid].metadata.get('bm25_score', -1e9):
                    best[uid] = d
            return sorted(best.values(), key=lambda d: d.metadata.get('bm25_score', 0.0), reverse=True)[:top_k]

        if hasattr(self, 'retrievers') and isinstance(self.retrievers, dict):
            src = detect(query) if query.strip() else 'en'
            src = src if src in ('en', 'de') else 'en'
            bag = []
            translator = getattr(self, 'translator', None)

            for lang, ret in self.retrievers.items():
                qq = translator.translate(query, lang) if translator and lang != src else query
                docs_with_scores = self._get_docs_with_scores(ret, qq, top_k)
                for doc, score in docs_with_scores:
                    doc.metadata['bm25_score'] = float(score)
                    bag.append(doc)

            best = {}
            for d in bag:
                uid = d.metadata.get('chunk_id') or d.metadata.get('record_id')
                if uid is None:
                    continue
                if uid not in best or d.metadata.get('bm25_score', -1e9) > best[uid].metadata.get('bm25_score', -1e9):
                    best[uid] = d

            return sorted(best.values(), key=lambda d: d.metadata.get('bm25_score', 0.0), reverse=True)[:top_k]

        raise AttributeError('Unsupported BilingualBM25 object format.')

class QEBM25:
    @staticmethod
    def _expand_query(query: str, base_retriever, fb_docs: int = 5, fb_terms: int = 5) -> str:
        def tok(text: str):
            try:
                return nltk.word_tokenize(text.lower())
            except Exception:
                return text.lower().split()

        hits = base_retriever.search(query, top_k=fb_docs)
        tokens = [
            t for h in hits for t in tok(h.page_content)
            if t.isalpha() and t not in STOP_EN and t not in STOP_DE
        ]
        extra = ' '.join(w for w, _ in nltk.FreqDist(tokens).most_common(fb_terms))
        return f'{query} {extra}' if extra else query

    def search(self, query: str, top_k: int = 100):
        if hasattr(self, 'base'):
            expanded = self._expand_query(query, self.base)
            return self.base.search(expanded, top_k)
        raise AttributeError('QEBM25 object missing base retriever.')

with open(PATH_BM25_PICKLE, 'rb') as f:
    bm25_raw = pickle.load(f)

class BM25RetrieverAdapter:
    def __init__(self, obj):
        self.obj = obj

    def search(self, query: str, top_k: int = 100):
        # Primary path
        if hasattr(self.obj, 'search'):
            try:
                return self.obj.search(query, top_k=top_k)
            except TypeError:
                return self.obj.search(query, k=top_k)

        # LangChain retriever fallback
        if hasattr(self.obj, 'invoke'):
            old_k = getattr(self.obj, 'k', None)
            if old_k is not None:
                self.obj.k = top_k
            docs = self.obj.invoke(query)
            if old_k is not None:
                self.obj.k = old_k
            for rank, d in enumerate(docs, start=1):
                if hasattr(d, 'metadata'):
                    d.metadata.setdefault('bm25_score', float(top_k - rank))
            return docs[:top_k]

        raise AttributeError(f'Unsupported BM25 object type: {type(self.obj)}')

bm25_retriever = BM25RetrieverAdapter(bm25_raw)
print('BM25 loaded:', type(bm25_raw), '-> adapter ready')


BM25 loaded: <class '__main__.BilingualBM25'> -> adapter ready


## 2.2 Retrieval Component: Dense Retriever

Initializes the multilingual E5 + Chroma retriever and returns dense candidates through the same interface used by other retrievers.



In [72]:
# Dense retriever
class DenseRetriever:
    def __init__(self, index_dir: pathlib.Path, model_name='intfloat/multilingual-e5-large-instruct', k: int = 100):
        self.k = k
        self.embeddings = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={'device': 'cuda' if os.path.exists('/proc/driver/nvidia/version') else 'cpu'},
            encode_kwargs={'batch_size': 32, 'normalize_embeddings': True},
        )
        self.store = Chroma(persist_directory=str(index_dir), embedding_function=self.embeddings)

    def _prep(self, q: str) -> str:
        return 'query: ' + q.strip()

    def search(self, query: str, top_k: int = 100):
        k = top_k or self.k
        hits = self.store.similarity_search_with_score(self._prep(query), k=k)
        out = []
        for doc, dist in hits:
            doc.metadata['dense_score'] = 1.0 - float(dist)
            out.append(doc)
        return out

dense_retriever = DenseRetriever(PATH_DENSE_INDEX, k=100)
print('Dense retriever ready.')


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Dense retriever ready.


## 2.3 Retrieval Component: GraphRAG Retriever

Loads GraphRAG resources and returns graph-informed candidates by selecting relevant communities and ranking chunks within them.



In [73]:
# GraphRAG retriever
class GraphRAGRetriever:
    def __init__(self, graph_root: pathlib.Path, chunk_pkl: pathlib.Path):
        self.root = graph_root
        self.emb_dir = graph_root / 'embeddings'
        self.chunk_pkl = chunk_pkl
        self.embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
        self._emb_cache = {}
        self._cid_cache = {}
        self._chunk_by_id = None
        self._chunk_vec_cache = {}
        self.comm2chunk = json.loads((self.root / 'comm2chunk_fixed.json').read_text(encoding='utf-8'))

    def _load_embeddings(self, level: int):
        if level in self._emb_cache:
            return self._emb_cache[level], self._cid_cache[level]
        mat = np.load(self.emb_dir / f'EMB_fixed_C{level}.npy')
        cid = json.loads((self.emb_dir / f'CID_fixed_C{level}.json').read_text(encoding='utf-8'))
        self._emb_cache[level] = mat
        self._cid_cache[level] = cid
        return mat, cid

    def _load_chunks(self):
        if self._chunk_by_id is not None:
            return self._chunk_by_id
        with open(self.chunk_pkl, 'rb') as f:
            docs_norm = pickle.load(f)

        def restore(d):
            raw = d.metadata.get('original_text') or d.page_content
            return Document(page_content=raw, metadata=d.metadata)

        docs = [restore(d) for d in docs_norm]
        self._chunk_by_id = {d.metadata['chunk_id']: d for d in docs}
        return self._chunk_by_id

    def _chunk_vec(self, cid: str, chunks: dict):
        if cid not in self._chunk_vec_cache:
            self._chunk_vec_cache[cid] = self.embedder.encode([chunks[cid].page_content], normalize_embeddings=True)[0]
        return self._chunk_vec_cache[cid]

    def retrieve(self, query: str, level: str = 'C1', k_comms: int = 24, top_k: int = 100):
        L = int(level.lstrip('C'))
        emb_mat, cid_list = self._load_embeddings(L)
        chunks = self._load_chunks()

        q_vec = self.embedder.encode([query], normalize_embeddings=True)[0]
        sims_comm = emb_mat @ q_vec
        best_idx = sims_comm.argsort()[::-1][:k_comms]

        cand_ids = set()
        for idx in best_idx:
            cand_ids.update(self.comm2chunk.get(cid_list[idx], []))

        scored = []
        for cid in cand_ids:
            if cid not in chunks:
                continue
            sim = float(self._chunk_vec(cid, chunks) @ q_vec)
            scored.append((cid, sim))

        scored.sort(key=lambda x: x[1], reverse=True)
        scored = scored[:top_k]

        out = []
        for cid, sim in scored:
            d = chunks[cid]
            d.metadata['grag_score'] = (sim + 1.0) / 2.0
            out.append(d)
        return out

    def search(self, query: str, top_k: int = 100):
        return self.retrieve(query=query, level='C1', k_comms=24, top_k=top_k)

graph_retriever = GraphRAGRetriever(PATH_GRAG_ROOT, PATH_CHUNK_PKL)
print('GraphRAG retriever ready.')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GraphRAG retriever ready.


In [74]:
# Shared utilities used by Step 2 agents
from collections import defaultdict
from typing import Any

def _uid(doc: Any):
    meta = getattr(doc, 'metadata', {}) or {}
    return meta.get('chunk_id') or meta.get('record_id') or meta.get('doc_id')

def _safe_unique(docs):
    out, seen = [], set()
    for d in docs:
        u = _uid(d)
        if u is None or u in seen:
            continue
        seen.add(u)
        out.append(d)
    return out

def _rrf_fuse(runs: dict, k_rrf: int = 60, weights=None):
    weights = weights or {'bm25': 1.2, 'dense': 1.0, 'graph': 0.6}
    scores = defaultdict(float)
    store = {}
    for name, docs in runs.items():
        w = float(weights.get(name, 1.0))
        for rank, d in enumerate(docs, start=1):
            u = _uid(d)
            if u is None:
                continue
            store.setdefault(u, d)
            scores[u] += w * (1.0 / (k_rrf + rank))
    fused = sorted(store.values(), key=lambda d: scores[_uid(d)], reverse=True)
    for d in fused:
        d.metadata['fused_score'] = float(scores[_uid(d)])
    return fused

def _token_set(text: str):
    text = (text or '').lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    return set(t for t in text.split() if t)

def _overlap_rerank(docs, query: str, top_k: int):
    q_terms = set(t.lower() for t in query.split() if t.strip())
    scored = []
    for d in docs:
        text = (d.metadata.get('original_text') or d.page_content or '').lower()
        overlap = len(q_terms & set(text.split())) / max(len(q_terms), 1)
        scored.append((overlap, d))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [d for _, d in scored[:top_k]]

print('Shared Step 2 utilities ready.')



Shared Step 2 utilities ready.


In [75]:
# Multi-agent role contracts
from dataclasses import dataclass, field
from typing import Dict, List, Any

@dataclass
class AgentState:
    query: str
    normalized_query: str = ''
    query_type: str = 'mixed'
    query_hints: Dict[str, float] = field(default_factory=dict)
    retrieval_by_agent: Dict[str, List[Any]] = field(default_factory=dict)
    fused_docs: List[Any] = field(default_factory=list)
    reranked_docs: List[Any] = field(default_factory=list)
    final_answer: str = ''
    evidence_ids: List[str] = field(default_factory=list)
    critic_ok: bool = False
    critic_feedback: str = ''
    needs_reretrieval: bool = False

class BaseAgent:
    name = 'base'
    def run(self, state: AgentState, **kwargs) -> AgentState:
        raise NotImplementedError

print('Agent contracts ready.')



Agent contracts ready.


In [76]:
# 1) Query Understanding Agent
class QueryUnderstandingAgent(BaseAgent):
    name = 'query_understanding'

    def run(self, state: AgentState, **kwargs) -> AgentState:
        q = (state.query or '').strip()
        q_low = q.lower()

        # Lightweight normalization
        state.normalized_query = ' '.join(q.split())

        # Rule-based type hints (fast + deterministic for baseline)
        keyword_signals = {'exactly', 'define', 'list', 'when', 'where', 'who'}
        graph_signals = {'relationship', 'connected', 'dependency', 'impact', 'between'}

        kw_score = sum(1 for t in keyword_signals if t in q_low)
        gr_score = sum(1 for t in graph_signals if t in q_low)

        if gr_score >= 1:
            q_type = 'graph'
        elif kw_score >= 2:
            q_type = 'keyword'
        elif len(q.split()) >= 8:
            q_type = 'semantic'
        else:
            q_type = 'mixed'

        state.query_type = q_type
        state.query_hints = {
            'bm25': 1.2 if q_type in {'keyword', 'mixed'} else 0.9,
            'dense': 1.2 if q_type in {'semantic', 'mixed'} else 0.9,
            'graph': 1.3 if q_type == 'graph' else 0.8,
        }
        return state

print('Query Understanding Agent ready.')



Query Understanding Agent ready.


In [77]:
# 2) Retriever Agents (BM25, Dense, GraphRAG)
class BM25RetrieverAgent(BaseAgent):
    name = 'bm25_retriever'
    def __init__(self, retriever):
        self.retriever = retriever
    def run(self, state: AgentState, top_k: int = 30, **kwargs) -> AgentState:
        state.retrieval_by_agent['bm25'] = _safe_unique(self.retriever.search(state.normalized_query, top_k=top_k))
        return state

class DenseRetrieverAgent(BaseAgent):
    name = 'dense_retriever'
    def __init__(self, retriever):
        self.retriever = retriever
    def run(self, state: AgentState, top_k: int = 30, **kwargs) -> AgentState:
        state.retrieval_by_agent['dense'] = _safe_unique(self.retriever.search(state.normalized_query, top_k=top_k))
        return state

class GraphRetrieverAgent(BaseAgent):
    name = 'graph_retriever'
    def __init__(self, retriever):
        self.retriever = retriever
    def run(self, state: AgentState, top_k: int = 30, **kwargs) -> AgentState:
        state.retrieval_by_agent['graph'] = _safe_unique(self.retriever.search(state.normalized_query, top_k=top_k))
        return state

print('Retriever Agents ready.')



Retriever Agents ready.


In [78]:
# 3) Fusion Agent
class FusionAgent(BaseAgent):
    name = 'fusion'

    def run(self, state: AgentState, top_k: int = 30, **kwargs) -> AgentState:
        runs = {
            'bm25': state.retrieval_by_agent.get('bm25', []),
            'dense': state.retrieval_by_agent.get('dense', []),
            'graph': state.retrieval_by_agent.get('graph', []),
        }
        fused = _rrf_fuse(runs, weights=state.query_hints or None)
        state.fused_docs = _safe_unique(fused)[:top_k]
        return state

print('Fusion Agent ready.')



Fusion Agent ready.


In [79]:
# 4) Re-Ranker Agent
class ReRankerAgent(BaseAgent):
    name = 'reranker'

    def run(self, state: AgentState, top_k: int = 10, **kwargs) -> AgentState:
        # Reuse notebook baseline overlap reranker for deterministic Colab execution
        state.reranked_docs = _overlap_rerank(state.fused_docs, state.normalized_query, top_k=top_k)
        return state

print('Re-Ranker Agent ready.')



Re-Ranker Agent ready.


In [80]:
# 5) Answer Synthesizer Agent
class AnswerSynthesizerAgent(BaseAgent):
    name = 'answer_synthesizer'

    def _build_context(self, docs: List[Any], max_docs: int = 5) -> str:
        parts = []
        for d in docs[:max_docs]:
            txt = (d.metadata.get('original_text') or d.page_content or '').strip()
            if txt:
                parts.append(txt)
        return "\n\n".join(parts)

    def run(self, state: AgentState, **kwargs) -> AgentState:
        ctx = self._build_context(state.reranked_docs or state.fused_docs)
        q = state.normalized_query

        # Extractive fallback (no paid API dependency)
        if not ctx:
            state.final_answer = 'No supporting context was retrieved.'
            state.evidence_ids = []
            return state

        sents = re.split(r'(?<=[.!?])\s+', ctx)
        q_terms = set(_token_set(q))
        scored = []
        for s in sents:
            st = set(_token_set(s))
            score = len(q_terms & st)
            scored.append((score, s.strip()))
        scored.sort(key=lambda x: x[0], reverse=True)

        best = [s for score, s in scored[:3] if s]
        state.final_answer = ' '.join(best) if best else 'Insufficient evidence for a grounded answer.'
        state.evidence_ids = [_uid(d) for d in (state.reranked_docs or state.fused_docs)[:5] if _uid(d) is not None]
        return state

print('Answer Synthesizer Agent ready.')



Answer Synthesizer Agent ready.


In [81]:
# 6) Critic Agent
class CriticAgent(BaseAgent):
    name = 'critic'

    def run(self, state: AgentState, min_support_overlap: float = 0.15, **kwargs) -> AgentState:
        answer_terms = _token_set(state.final_answer)
        if not answer_terms:
            state.critic_ok = False
            state.needs_reretrieval = True
            state.critic_feedback = 'Answer is empty.'
            return state

        support_docs = state.reranked_docs or state.fused_docs
        support_text = ' '.join((d.metadata.get('original_text') or d.page_content or '') for d in support_docs[:5])
        support_terms = _token_set(support_text)
        overlap = len(answer_terms & support_terms) / max(len(answer_terms), 1)

        state.critic_ok = overlap >= min_support_overlap
        state.needs_reretrieval = not state.critic_ok
        state.critic_feedback = (
            f'Support overlap={overlap:.3f}; threshold={min_support_overlap:.3f}. '
            + ('Grounded.' if state.critic_ok else 'Potentially ungrounded: trigger re-retrieval.')
        )
        return state

print('Critic Agent ready.')



Critic Agent ready.


In [82]:
# 7) Orchestrator-ready linear execution (can be replaced by external orchestrator)
class MultiAgentPipeline:
    def __init__(self):
        self.query_agent = QueryUnderstandingAgent()
        self.bm25_agent = BM25RetrieverAgent(bm25_retriever)
        self.dense_agent = DenseRetrieverAgent(dense_retriever)
        self.graph_agent = GraphRetrieverAgent(graph_retriever)
        self.fusion_agent = FusionAgent()
        self.reranker_agent = ReRankerAgent()
        self.answer_agent = AnswerSynthesizerAgent()
        self.critic_agent = CriticAgent()

    def run(self, query: str, retrieve_k: int = 30, top_k: int = 10, retry_once: bool = True):
        state = AgentState(query=query)

        # Step A: understanding
        state = self.query_agent.run(state)

        # Step B: retrieval (all specialized retrievers)
        state = self.bm25_agent.run(state, top_k=retrieve_k)
        state = self.dense_agent.run(state, top_k=retrieve_k)
        state = self.graph_agent.run(state, top_k=retrieve_k)

        # Step C: fusion -> rerank -> synthesis -> critique
        state = self.fusion_agent.run(state, top_k=retrieve_k)
        state = self.reranker_agent.run(state, top_k=top_k)
        state = self.answer_agent.run(state)
        state = self.critic_agent.run(state)

        # Optional single retry hook (external orchestrator can override with advanced routing)
        if retry_once and state.needs_reretrieval:
            boosted = dict(state.query_hints)
            boosted['graph'] = boosted.get('graph', 1.0) + 0.3
            boosted['dense'] = boosted.get('dense', 1.0) + 0.2
            state.query_hints = boosted
            state = self.fusion_agent.run(state, top_k=retrieve_k)
            state = self.reranker_agent.run(state, top_k=top_k)
            state = self.answer_agent.run(state)
            state = self.critic_agent.run(state)

        return state

pipeline = MultiAgentPipeline()
print('Multi-agent pipeline ready.')



Multi-agent pipeline ready.


In [85]:
# Example usage
sample_query = 'Who was president of ETH in 2003?'
out = pipeline.run(sample_query, retrieve_k=10, top_k=5, retry_once=False)

print('Query type:', out.query_type)
print('Critic OK:', out.critic_ok)
print('Critic feedback:', out.critic_feedback)
print('Evidence IDs:', out.evidence_ids[:5])
print('\nFinal answer:\n', out.final_answer)



Query type: mixed
Critic OK: True
Critic feedback: Support overlap=1.000; threshold=0.150. Grounded.
Evidence IDs: ['44b952fdee9be9bcd573f8131a26f3004bb7abb0_fixed_1', '6398e26197c3c3b80c214556e60b53b6c4950fa5_fixed_6', '7dbea25051647a6a29ea919047b488fb43be6bd3_fixed_0', '7f9dfcdad9934d0dd7a07f3651d490b7dc13071b_fixed_9', '8ce837c16e7dad4f4db2f57facd68f83f1b8fa6d_fixed_0']

Final answer:
 from 1987 to 1990, he was president of the swiss school board, and from 1990 to 1997 director of the swiss science agency (the predecessor organisation of today's state secretariat for education research and innovation (seri) - from 1992 in the capacity of state secretary. former eth president heinrich ursprung deceased: appointments from 6,700 applications to the eth board over his 14 years in office. as a great supporter of scientific networks, ursprung contributed to the formulation of the eth act that was drafted at the time.


In [87]:
# Retriever debug view: inspect top results before and after fusion

def _preview_docs(name, docs, n=3, max_chars=220):
    print(f"\n[{name}] top {min(n, len(docs))} / {len(docs)}")
    for i, d in enumerate(docs[:n], start=1):
        uid = _uid(d)
        text = (d.metadata.get('original_text') or d.page_content or '').replace('\n', ' ').strip()
        text = text[:max_chars] + ('...' if len(text) > max_chars else '')
        score_keys = [k for k in ('bm25_score', 'dense_score', 'grag_score', 'fused_score') if k in d.metadata]
        score_str = ', '.join(f"{k}={d.metadata.get(k):.4f}" for k in score_keys)
        print(f"{i}. id={uid} | {score_str}")
        print(f"   {text}")


def debug_query(query, retrieve_k=20, top_k=10):
    print('Query:', query)

    # Individual retrievers
    bm = _safe_unique(bm25_retriever.search(query, top_k=retrieve_k))
    de = _safe_unique(dense_retriever.search(query, top_k=retrieve_k))
    gr = _safe_unique(graph_retriever.search(query, top_k=retrieve_k))

    _preview_docs('BM25', bm)
    _preview_docs('Dense', de)
    _preview_docs('GraphRAG', gr)

    # Full pipeline outputs
    out = pipeline.run(query, retrieve_k=retrieve_k, top_k=top_k, retry_once=False)
    _preview_docs('Fused', out.fused_docs)
    _preview_docs('Re-ranked', out.reranked_docs)

    print('\nQuery type:', out.query_type)
    print('Critic OK:', out.critic_ok)
    print('Critic feedback:', out.critic_feedback)
    print('Evidence IDs:', out.evidence_ids)
    print('\nFinal answer:\n', out.final_answer)

# Example debug call:
debug_query('Who was president of ETH in 2003?', retrieve_k=20, top_k=10)



Query: Who was president of ETH in 2003?

[BM25] top 3 / 20
1. id=6398e26197c3c3b80c214556e60b53b6c4950fa5_fixed_6 | bm25_score=18.1182, fused_score=0.0197
   blog ukrainekrieg zurueck ins 19 jahrhundert: pon. 2022. "the future is history: restorative nationalism and conflict in post-napoleonic europe." eth zuerich. 8 [externe seite on the historical unity of russians and ukra...
2. id=db746e21e0320563c91208ca5f547c10b17f74f7_fixed_0 | bm25_score=16.7903, fused_score=0.0194
   nachhaltigkeit naeherbringen und weitergeben: engagiert fuer die zukunft ein eth-studium oeffnet viele tueren. und doch kommen immer wieder fragen auf wie: welche berufsentscheidungen treffen wir? wie koennen wir das stu...
3. id=7f9dfcdad9934d0dd7a07f3651d490b7dc13071b_fixed_9 | bm25_score=16.2253, fused_score=0.0190
   zwoelf professorinnen und professoren ernannt: dr. sylke poehling** (*1967), global head und senior vice president fuer therapeutic modalities bei roche pharma research &amp; early development (r